# GDELT co-mention crosswalk + edge-confidence weighting (Extension)

Goal: use GDELT GKG (Global Knowledge Graph) news co-mention frequency between
firm pairs as an edge-confidence weight on the supply-chain graph, on top of
the existing binary supplier/customer edges from WRDS Compustat Segment data.

**Why this notebook exists / what it can and can't do:**
- I (Claude) cannot pull raw GDELT GKG data myself in this environment -- the
  GKG feed is distributed as ~15-minute-interval gzipped CSVs (or via Google
  BigQuery's `gdelt-bq` public dataset), both of which require either bulk
  HTTP download tooling or BigQuery credentials that aren't available here.
  WebFetch only extracts readable text from a single page, not bulk data files.
- What I *can* prepare: the ticker/company-name -> gvkey crosswalk, which is
  the other missing piece (confirmed no local file has this -- checked
  `scg_edges_sec_partners.parquet`, `crsp_monthly.parquet`, and the whole
  `AB IP` folder; nothing has a clean ticker column joined to gvkey).
- **Cell 1 is run by the user** (interactive WRDS password prompt / `.pgpass`),
  same pattern as `factset_revere_extraction.ipynb`. Every cell after that
  reuses the `db` connection object it creates.

**After running this notebook**, the user still needs to separately obtain
GDELT GKG data (e.g. via Google BigQuery's public `gdelt-bq.gdeltv2.gkg`
table, which supports SQL filtering by date/theme and is far more tractable
than downloading the raw 15-minute files) and run co-mention counts through
the `crosswalk_ticker_to_gvkey.parquet` produced here.

In [ ]:
# SQLAlchemy 2.x compatibility patch (identical to factset_revere_extraction.ipynb) --
# the installed wrds package (3.1.x) predates SQLAlchemy 2.x's stricter
# Connection.execute() API, which requires raw SQL strings to be wrapped in
# sqlalchemy.text(...) rather than passed bare. Guarded so re-running this
# cell doesn't double-wrap and recurse infinitely.
import sqlalchemy as sa
from sqlalchemy.engine import Connection

if not getattr(Connection.execute, "_is_sa2_compat_patch", False):
    _original_execute = Connection.execute

    def _execute_compat(self, statement, *args, **kwargs):
        if isinstance(statement, str):
            statement = sa.text(statement)
        return _original_execute(self, statement, *args, **kwargs)

    _execute_compat._is_sa2_compat_patch = True
    Connection.execute = _execute_compat

import wrds

# Run this cell yourself -- it will prompt for your WRDS username/password
# (or use the .pgpass file already saved from earlier).
db = wrds.Connection()

## 1. Pull the ticker -> permno -> gvkey crosswalk

`wrdsapps_link_crsp_factset.fscrsplink` has both `ticker` and `permno` columns
(confirmed present during the FactSet Revere extraction earlier). This gives a
clean, non-fuzzy path from GDELT's organization mentions (once resolved to
tickers) to this project's gvkey system, joined through the existing
`data/ccm_link.parquet` permno->gvkey crosswalk.

In [ ]:
query = """
    SELECT DISTINCT ticker, permno, cusip
    FROM wrdsapps_link_crsp_factset.fscrsplink
    WHERE ticker IS NOT NULL
"""
ticker_permno = db.raw_sql(query)
print(ticker_permno.shape)
ticker_permno.head()

In [ ]:
import pandas as pd

# Map permno -> gvkey via the existing CCM link table (same file used for
# FactSet Revere permno->gvkey resolution)
ccm = pd.read_parquet('../data/ccm_link.parquet')
ccm = ccm[ccm['linktype'].isin(['LU', 'LC']) & ccm['linkprim'].isin(['P', 'C'])]
ccm_small = ccm[['permno', 'gvkey']].drop_duplicates()

crosswalk = ticker_permno.merge(ccm_small, on='permno', how='inner')
crosswalk = crosswalk.dropna(subset=['ticker', 'gvkey']).drop_duplicates(subset=['ticker', 'gvkey'])
print(f'{len(crosswalk)} ticker-gvkey pairs resolved out of {ticker_permno["permno"].nunique()} distinct permnos')
crosswalk.head()

In [ ]:
# Restrict to gvkeys that actually appear in our features panel -- no point
# keeping crosswalk rows for firms outside the replication's firm universe.
features_gvkeys = pd.read_csv('../data/features.csv', usecols=['gvkey'])['gvkey'].astype(str).unique()
crosswalk['gvkey'] = crosswalk['gvkey'].astype(str)
crosswalk_in_universe = crosswalk[crosswalk['gvkey'].isin(features_gvkeys)]
print(f'{len(crosswalk_in_universe)} crosswalk rows within the {len(features_gvkeys)}-gvkey replication universe')

crosswalk_in_universe.to_parquet('../data/crosswalk_ticker_to_gvkey.parquet', index=False)
print('Saved: ../data/crosswalk_ticker_to_gvkey.parquet')

## 2. Next steps (not run in this notebook)

1. Obtain GDELT GKG data for the paper's sample window. The practical path is
   Google BigQuery's public dataset `gdelt-bq.gdeltv2.gkg` (or the v1 `gdelt-bq.gdeltv2.events`),
   queried by date range and filtered to rows where `V2Organizations` contains
   two or more of our tracked tickers/company names in the same article.
2. For each (firm_i, firm_j, month) triple, count co-mentions in that month.
3. Normalize counts into an edge-confidence weight in, e.g., [0, 1] (min-max or
   rank-normalize per month, consistent with how the 64 firm characteristics
   are already rank-normalized in this pipeline).
4. In `GNN model.ipynb`, `prepare_graph_data` (cell 11) currently builds a
   plain unweighted `edge_index`. `TransformerConv` supports `edge_attr` --
   the co-mention weight would be passed as `edge_attr` and the model's
   `forward()` (cell 10) updated to pass it through:
   `layer(x, edge_index, edge_attr)`.
5. Retrain and compare Table 5 Sharpe ratios against the unweighted-edge
   baseline to see whether news co-mention confidence improves the GNN's
   long-short spanning-regression Sharpe ratio.

This notebook stops at step 1 (the crosswalk) since steps 2 onward require
GDELT data access this environment does not have.